# SemanticSCVI hyperparameter sweep — CD4 CTCL atlas

Parallel to `four_way_benchmark_haniffa_cd8_genept_sweep.ipynb`, but for the **CD4 non-Treg atlas** model of `18_semantic_cd4_atlas_malignancy.ipynb` (geometric, **Geneformer** prior). Preprocessing is identical to nb18 (CD4 non-Treg subset, ribo drop, **Geneformer-vocab genes only**, HVG).

**Flow (like `17_mrvi_pseudosample_malignancy.ipynb`):** this notebook runs the light preprocessing on the interactive kernel, writes a slim input + config, then **submits a bsub job from a cell** (`jobs/run_cd4_sweep.sh` → `jobs/run_cd4_sweep.py`) that does the heavy GPU work (train every variant + projection benchmark + report). Poll from a cell; display the HTML report when done.

- Sweeps **5 axes**: `max_epochs`, `warmup_epochs`, `n_epochs_kl_warmup`, `coherence_weight`, `n_layers`. Edit `BASELINE` (fixed defaults) and `SWEEP` (per-axis alternatives). Each value in `SWEEP[axis]` makes one variant differing from baseline **only** in that axis; baseline always included; duplicates deduped.
- **Projections only** (no gene clustering); top-30 loadings per projection.
- LLM scoring with **Opus only**, judging each projection in the context of CD4 T cells spanning benign/malignant states in T-cell lymphoma.
- Per-projection **MSigDB** enrichment with `lib1_immune.gmt` + `lib2_cd4.gmt`.
- One-third subsample of CD4 cells for speed.

**Cells:** config → imports/harness → data prep (nb18-verbatim) → subsample → build `lib2_cd4.gmt` → **write job input** → **submit** → **poll** → **display report**.

In [ ]:
# ============================================================
# Parameters — TCR rule + SemanticSCVI sweep config (geometric Geneformer, from nb18).
# ============================================================
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data starting from {start}")


NB_DIR = _resolve_nb_dir()
print(f"NB_DIR = {NB_DIR}")

# ---- paths (same atlas inputs as nb18) ----
OBJ            = NB_DIR / "data" / "atlas_joint" / "skin_T_annotated.h5ad"
TCR_OBJ        = NB_DIR / "data" / "atlas_joint" / "skin_T_tcr_annotated.h5ad"
LI_TCR         = NB_DIR / "data" / "Li2024_atlas" / "li2024_tcr_malignancy.parquet"
GENE_ID_SOURCE = NB_DIR / "data" / "cache" / "cnmf_malignant_counts.h5ad"   # gene_name -> gene_id
SEMANTIC_CACHE_GENEFORMER = NB_DIR / "data" / "mf_cd4_atlas_geneformer.pt"
MODEL_CACHE_DIR  = NB_DIR / "models" / ".model_cache_semantic_geom_cd4_atlas"
SEM_CACHE_PARENT = "semantic_scvi_sweep"
OUT_DIR          = NB_DIR / "benchmark_results" / "cd4_atlas_semantic_sweep"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# MSigDB libraries (in notebooks/, one level up from MF/)
LIB1_GMT = NB_DIR.parent / "lib1_immune.gmt"
LIB2_GMT = NB_DIR.parent / "lib2_cd4.gmt"
C7_SOURCE = Path("/home/projects/nyosef/zvise/metrics/pathways/c7.all.v2025.1.Hs.symbols.gmt")
PATHWAY_INDEX = NB_DIR.parent / "pathway_index.pkl"

# ---- bsub job handoff (slim input + config consumed by jobs/run_cd4_sweep.py) ----
# Job input lives in OUT_DIR/_job_input/ which the report-cleanup step preserves
# (alongside _llm_cache / _score_cache), so resubmits don't lose it.
JOBS_DIR      = NB_DIR / "jobs"
JOB_INPUT_DIR = OUT_DIR / "_job_input"
CD4_INPUT     = JOB_INPUT_DIR / "cd4_sweep_input.h5ad"
SEM_MAP_PATH  = JOB_INPUT_DIR / "cd4_sweep_semantic_map.pt"
JOB_CONFIG    = JOBS_DIR / "cd4_sweep_config.json"
JOB_SH        = JOBS_DIR / "run_cd4_sweep.sh"
JOB_LOG       = JOBS_DIR / "run_cd4_sweep.bsub.log"

# ---- TCR dominant-clone rule (relaxed: fold-change >= 1.33) ----
FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN = 0.05, 1.33, 2

# ---- CD4 non-Treg selection (cell_type_T) ----
CD4_T_TYPES = ["CD4_Cm", "CD4_Th2", "CD4_cytotoxic", "CD4_mem"]

# ---- Preprocessing ----
HVG_TOP_N  = 2500
HVG_FLAVOR = "seurat_v3"
SUBSAMPLE_FRACTION = 1.0 / 3.0   # one-third of CD4 cells for the sweep

# ---- Shared model size / batch ----
N_LATENT  = 10
BATCH_KEY = "sample_id"

# ---- Projection scoring ----
PER_PROJECTION_N_TOP = 30

# ---- Sweep: baseline = nb18 geometric Geneformer config ----
BASELINE = {
    "max_epochs":         200,
    "warmup_epochs":       20,
    "n_epochs_kl_warmup": 100,
    "coherence_weight":  1000.0,
    "n_layers":             1,
}
# Per-axis alternatives. Each value -> one variant differing only in that axis.
SWEEP = {
    "max_epochs":         [],
    "warmup_epochs":      [],
    "n_epochs_kl_warmup": [],
    "coherence_weight":   [500.0, 2000.0,3000.0],
    "n_layers":           [],
}
# Fixed (non-swept) SemanticSCVI constructor kwargs (from nb18).
# coherence_weight / n_layers are swept; max_epochs/warmup/kl are passed explicitly.
FIXED_KWARGS = dict(
    loss_mode="geometric",
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

# Per-variant cache_dir is keyed on a hash of the ABSOLUTE params (see _cache_slug
# in the next cell), so changing BASELINE never collides: same config -> cache hit,
# any change -> fresh train. No need to clear the cache when editing BASELINE.

In [ ]:
import sys, gc, json, hashlib, subprocess, importlib
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))            # MF/ holds atlas_join_helpers, skin_T_cnv_helpers
if str(NB_DIR.parent) not in sys.path:
    sys.path.insert(0, str(NB_DIR.parent))     # notebooks/ holds benchmark_helpers

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import scvi

import atlas_join_helpers as H
import skin_T_cnv_helpers as C
importlib.reload(H); importlib.reload(C)
from benchmark_helpers import get_or_build_geneformer_map   # training/benchmark run in jobs/run_cd4_sweep.py

scvi.settings.seed = 0
np.random.seed(0)
sc.settings.verbosity = 1
print("scvi", scvi.__version__, "| scanpy", sc.__version__, "| CUDA:", torch.cuda.is_available())

# ---- generalized sweep harness (all axes in BASELINE are sweepable) ----
_AXES = tuple(BASELINE)
_AXIS_SHORT = {"max_epochs": "me", "warmup_epochs": "w", "n_epochs_kl_warmup": "kl",
               "coherence_weight": "cw", "n_layers": "nl"}


def _fmt_val(v):
    if isinstance(v, float):
        return str(int(v)) if v.is_integer() else f"{v:g}".replace(".", "p")
    return str(v)


def _variant_name(variant, baseline):
    """Human-readable label = diff from baseline (used in the report)."""
    diffs = [k for k in _AXES if variant[k] != baseline[k]]
    if not diffs:
        return "sem_baseline"
    return "sem_" + "_".join(f"{_AXIS_SHORT[k]}{_fmt_val(variant[k])}" for k in diffs)


def _cache_slug(variant, n=10):
    """Stable 10-char hash of the ABSOLUTE params that define the trained model.
    cache_dir is keyed on this (not the diff-name), so changing BASELINE never
    collides: same config -> cache hit, any change -> fresh train."""
    blob = json.dumps(
        {
            "axes": {k: variant[k] for k in _AXES},
            "fixed": dict(sorted(FIXED_KWARGS.items())),
            "batch_key": BATCH_KEY,
            "labels_key": "cell_type_T",
            "hvg_top_n": HVG_TOP_N,
        },
        default=str, sort_keys=True,
    )
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


def _expand(baseline, sweep):
    for k in sweep:
        if k not in _AXES:
            raise ValueError(f"SWEEP key {k!r} not sweepable. Allowed: {_AXES}.")
    seen, variants = set(), []
    def _add(v):
        t = tuple(sorted((k, v[k]) for k in _AXES))
        if t not in seen:
            seen.add(t)
            variants.append(dict(v))
    _add(baseline)
    for axis, values in sweep.items():
        for val in values:
            v = dict(baseline); v[axis] = val
            _add(v)
    return variants


variants = _expand(BASELINE, SWEEP)
print(f"{len(variants)} variant(s):")
for v in variants:
    print(f"  {_variant_name(v, BASELINE):>20s}  [{_cache_slug(v)}]  " +
          "  ".join(f"{_AXIS_SHORT[k]}={v[k]}" for k in _AXES))

In [ ]:
adata = C.build_or_load_tcr_object(OBJ, TCR_OBJ, LI_TCR, H)   # cached ~12 GB object
is_li = adata.obs["study"].astype(str).eq("li2024").to_numpy()
print("loaded:", adata.shape)

In [ ]:
# unified TRB-primary clone key + per-donor dominant-clone malignancy (relaxed fold >= 1.33)
dom_tbl = C.recompute_dominant_clone(adata, H, is_li, FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN)
print(dom_tbl.to_string(index=False))

clone_summary = C.clone_summary_table(adata, FRAC_THRESH, RATIO_THRESH)
print(f"\nthresholds: dom_frac >= {FRAC_THRESH}  AND  fold_change >= {RATIO_THRESH}")
print(f"malignant donors: {int(clone_summary['malignant'].sum())}/{len(clone_summary)}")
print("\nTCR-malignant cells by study:\n",
      adata.obs.groupby("study", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))
clone_summary

In [ ]:
# drop samples (donors) with no TCR at all — malignancy is undefined there
tcr_donors = adata.obs.loc[adata.obs["has_tcr"].astype(bool), "donor"].unique()
dropped = sorted(set(adata.obs["donor"].unique()) - set(tcr_donors))
print(f"donors with TCR: {len(tcr_donors)} | dropped (no TCR): {len(dropped)} -> {dropped}")
adata = adata[adata.obs["donor"].isin(tcr_donors)].copy()
print("after drop:", adata.shape)
print(adata.obs.groupby("study", observed=True)["has_tcr"].agg(["size", "sum"]))

In [ ]:
adata_cd4 = adata[adata.obs["cell_type_T"].isin(CD4_T_TYPES)].copy()

# drop ribosomal protein genes (RPS*/RPL*) — they dominate loadings and are uninformative here
ribo = adata_cd4.var_names.str.upper().str.startswith(("RPS", "RPL"))
print(f"removing {int(ribo.sum())} ribosomal protein genes (RPS*/RPL*)")
adata_cd4 = adata_cd4[:, ~ribo].copy()

adata_cd4.X = adata_cd4.layers["raw_counts"].copy()   # NB likelihood needs raw counts
print("CD4 non-Treg:", adata_cd4.shape)
print("cell_type_T:\n", adata_cd4.obs["cell_type_T"].value_counts())
print("\ntcr_is_malignant:", int(adata_cd4.obs["tcr_is_malignant"].astype(bool).sum()),
      "/", adata_cd4.n_obs)
print("by study:\n",
      adata_cd4.obs.groupby("study", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))
del adata; gc.collect()

In [ ]:
src = sc.read_h5ad(GENE_ID_SOURCE, backed="r")
sym2ens = dict(zip(src.var["gene_name"].astype(str), src.var["gene_id"].astype(str)))
adata_cd4.var["gene_id"] = [sym2ens.get(s, s) for s in adata_cd4.var_names.astype(str)]
n_mapped = int(sum(g.startswith("ENSG") for g in adata_cd4.var["gene_id"]))
print(f"gene_id mapped to Ensembl: {n_mapped}/{adata_cd4.n_vars}")
del src; gc.collect()

In [ ]:
# Build the FULL-gene Geneformer map first (cached). Then HVG-subset adata + map together.
# Delete the .pt to force a rebuild.
semantic_map = get_or_build_geneformer_map(
    adata_cd4, SEMANTIC_CACHE_GENEFORMER, var_id_key="gene_id",
)
print("semantic_map (raw):", tuple(semantic_map.shape))

# Stale-cache guard: rebuild if the cached map rows don't match the current full-gene adata.
if semantic_map.shape[0] != adata_cd4.n_vars:
    print(f"shape mismatch ({semantic_map.shape[0]} vs {adata_cd4.n_vars}) — rebuilding")
    SEMANTIC_CACHE_GENEFORMER.unlink()
    semantic_map = get_or_build_geneformer_map(
        adata_cd4, SEMANTIC_CACHE_GENEFORMER, var_id_key="gene_id",
    )
    print("semantic_map (rebuilt):", tuple(semantic_map.shape))

# Keep only genes with a real Geneformer prior (non-zero embedding row). Out-of-vocab
# genes (~all non-coding RP11-*/AC0*/LINC*/MIR*) are zero-filled by the map builder and
# carry no semantic signal — drop them BEFORE HVG so HVG ranks variance only among
# semantically-grounded (effectively protein-coding) genes.
in_vocab = (semantic_map.norm(dim=1) > 0).cpu().numpy()
print(f"in-vocab (non-zero Geneformer row): {int(in_vocab.sum())}/{adata_cd4.n_vars}")
adata_cd4 = adata_cd4[:, in_vocab].copy()
semantic_map = semantic_map[torch.as_tensor(in_vocab)]

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(adata_cd4, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False)
    keep = adata_cd4.var["highly_variable"].values
    adata_cd4 = adata_cd4[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
print("after HVG:", adata_cd4.shape, "| semantic_map:", tuple(semantic_map.shape))

In [ ]:
# Subsample to one-third of CD4 cells for the sweep (seeded; shared across all variants).
if SUBSAMPLE_FRACTION is not None and SUBSAMPLE_FRACTION < 1.0:
    n_before = adata_cd4.n_obs
    sc.pp.subsample(adata_cd4, fraction=SUBSAMPLE_FRACTION, random_state=0)
    print(f"subsampled {n_before} -> {adata_cd4.n_obs} cells (fraction={SUBSAMPLE_FRACTION:.3f})")
    print("tcr_is_malignant after subsample:",
          int(adata_cd4.obs["tcr_is_malignant"].astype(bool).sum()), "/", adata_cd4.n_obs)
else:
    print(f"no subsampling (n_obs={adata_cd4.n_obs})")

In [ ]:
# Build lib2_cd4.gmt once: curated HC_CD4_* block + CD4-filtered C7 IMMUNESIGDB sets.
import re

if LIB2_GMT.exists():
    print(f"lib2_cd4.gmt exists ({sum(1 for _ in LIB2_GMT.open())} sets) — skipping build")
else:
    HC_CD4 = {
        "HC_CD4_NAIVE":          ["CCR7", "SELL", "TCF7", "LEF1", "IL7R"],
        "HC_CD4_TH1":            ["TBX21", "IFNG", "CXCR3", "STAT1", "IL12RB2"],
        "HC_CD4_TH2":            ["GATA3", "IL4", "IL5", "IL13", "CCR4"],
        "HC_CD4_TH17":           ["RORC", "IL17A", "IL17F", "IL23R", "CCR6"],
        "HC_CD4_TREG":           ["FOXP3", "IL2RA", "CTLA4", "IKZF2", "TNFRSF18"],
        "HC_CD4_TFH":            ["BCL6", "CXCR5", "PDCD1", "ICOS", "IL21"],
        "HC_CD4_CYTOTOXIC":      ["GZMB", "PRF1", "NKG7", "GNLY", "GZMK"],
        "HC_CD4_EXHAUSTION":     ["TOX", "LAG3", "TIGIT", "HAVCR2", "PDCD1"],
        "HC_CD4_ACTIVATION":     ["CD69", "IL2RA", "TNFRSF9", "CD40LG", "ICOS"],
        "HC_CD4_MALIGNANT_CTCL": ["TOX", "GATA3", "TWIST1", "KIR3DL2", "PLS3"],
    }
    KEEP = re.compile(r"CD4|TH1|TH2|TH17|TREG|TFH|TCONV|THELPER|T_HELPER|TFOLLICULAR")
    DROP_CD8 = re.compile(r"CD8")
    c7_lines = []
    for line in C7_SOURCE.open():
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 3:
            continue
        name = parts[0]
        if not KEEP.search(name):
            continue
        if DROP_CD8.search(name) and "CD4" not in name:
            continue
        c7_lines.append(line.rstrip("\n"))
    with LIB2_GMT.open("w") as fh:
        for nm, genes in HC_CD4.items():
            fh.write("\t".join([nm, "NA", *genes]) + "\n")
        for line in c7_lines:
            fh.write(line + "\n")
    print(f"built {LIB2_GMT.name}: {len(HC_CD4)} curated + {len(c7_lines)} C7 = {len(HC_CD4)+len(c7_lines)} sets")

In [ ]:
# ---- write the slim job input (adata + semantic_map) + config for jobs/run_cd4_sweep.py ----
JOBS_DIR.mkdir(parents=True, exist_ok=True)
JOB_INPUT_DIR.mkdir(parents=True, exist_ok=True)

adata_cd4.write_h5ad(CD4_INPUT)
torch.save(semantic_map, SEM_MAP_PATH)
print(f"wrote input        -> {CD4_INPUT}  {adata_cd4.shape}")
print(f"wrote semantic_map -> {SEM_MAP_PATH}  {tuple(semantic_map.shape)}")

# name = report label (diff from baseline); cache_key = hash of absolute params.
job_variants = [
    {"name": _variant_name(v, BASELINE), "cache_key": _cache_slug(v), **{k: v[k] for k in _AXES}}
    for v in variants
]
notes = (
    f"SemanticSCVI sweep [Geneformer, CD4 CTCL atlas, projections-only]: {len(variants)} variant(s). "
    f"Fixed: n_latent={N_LATENT}, HVG_TOP_N={HVG_TOP_N}, SUBSAMPLE_FRACTION={SUBSAMPLE_FRACTION:.3f}, "
    f"PER_PROJECTION_N_TOP={PER_PROJECTION_N_TOP}. LLM judge: Opus only; MSigDB: lib1_immune + lib2_cd4."
)
cfg = {
    "input_h5ad":     str(CD4_INPUT),
    "semantic_map":   str(SEM_MAP_PATH),
    "model_cache_dir": str(MODEL_CACHE_DIR / SEM_CACHE_PARENT),
    "out_dir":        str(OUT_DIR),
    "lib1_gmt":       str(LIB1_GMT),
    "lib2_gmt":       str(LIB2_GMT),
    "pathway_index":  str(PATHWAY_INDEX),
    "batch_key":      BATCH_KEY,
    "labels_key":     "cell_type_T",
    "per_projection_n_top": PER_PROJECTION_N_TOP,
    "cell_type_context": ("CD4 T cells from cutaneous T-cell lymphoma (mycosis fungoides) skin, "
                          "spanning benign reactive and malignant clonal states"),
    "fixed_kwargs":   FIXED_KWARGS,
    "variants":       job_variants,
    "notes":          notes,
}
JOB_CONFIG.write_text(json.dumps(cfg, indent=2))
print(f"wrote job config   -> {JOB_CONFIG}  ({len(job_variants)} variants)")

In [ ]:
# ---- submit the sweep bsub job (train all variants + projection benchmark + report on GPU) ----
# Guard against double-submit: skip only if a cd4_sweep job is already PEND/RUN.
_q = subprocess.run(["bjobs", "-J", "cd4_sweep"], capture_output=True, text=True).stdout
if any(s in _q for s in (" RUN ", " PEND ")):
    print("a cd4_sweep job is already active; not resubmitting:\n", _q)
else:
    r = subprocess.run(["bash", str(JOB_SH)], capture_output=True, text=True)
    print(r.stdout, end=""); print(r.stderr, end="")
    print("\nlog:", JOB_LOG)
    print("poll the next cell (re-run) until a new report HTML appears, then run the display cell")

In [ ]:
# ---- poll: re-run until the report HTML exists; shows queue status + per-variant cache + log tail ----
print("=== bjobs -a (cd4_sweep) ===")
print(subprocess.run(["bjobs", "-a", "-J", "cd4_sweep"], capture_output=True, text=True).stdout or "(no jobs)")

reports = sorted(OUT_DIR.glob("cd4_atlas_semantic_sweep_*.html"))
print(f"\nreport HTML: {'OK ' + reports[-1].name if reports else '... not yet'}")
for v in variants:
    name, key = _variant_name(v, BASELINE), _cache_slug(v)
    mp = MODEL_CACHE_DIR / SEM_CACHE_PARENT / key / "model.pt"
    print(f"  {'OK ' if mp.exists() else '...'} {name}  [{key}]")

if JOB_LOG.exists():
    print("\n=== tail", JOB_LOG.name, "===")
    print(subprocess.run(["tail", "-n", "30", str(JOB_LOG)], capture_output=True, text=True).stdout)

In [ ]:
# ---- display the report once the job has finished ----
reports = sorted(OUT_DIR.glob("cd4_atlas_semantic_sweep_*.html"))
if not reports:
    print("no report yet — submit the job and poll until the HTML appears")
else:
    report = reports[-1]
    print(f"report: {report}  ({report.stat().st_size / 1e6:.1f} MB)")
    from IPython.display import IFrame, display
    display(IFrame(str(report.relative_to(NB_DIR)), width="100%", height=700))